In [12]:
import arcpy

In [ ]:
# ── PATHS ──────────────────────────────────────────────────────────────────────
csv_in     = r"C:\Users\kkeii\OneDrive\Documents\THESIS_REPOSITORIES\MISSION_REPOSITORIES\UVI2_Around_STX_20250306\data\4_data_interpolated\HF_background_INTERPOLATED_UVI2_Around_STX_20250306.csv"

gdb        = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Projects\THESIS_PROJECTS\map_setup_UVI2_Around_STX_20250306\map_setup_UVI2_Around_STX_20250306.gdb"
aprx_path  = r"C:\Users\kkeii\OneDrive\Documents\ArcGIS\Projects\THESIS_PROJECTS\map_setup_UVI2_Around_STX_20250306\map_setup_UVI2_Around_STX_20250306.aprx"

line_wgs84 = gdb + r"\HF_background_line_WGS84"
line_nad83 = gdb + r"\HF_background_line_NAD83"
pts_wgs84 = gdb + r"\HF_background_points_WGS84"
pts_nad83 = gdb + r"\calc_HF_background_points_NAD83"

geog_transform = "WGS_1984_(ITRF00)_To_NAD_1983"

aprx       = arcpy.mp.ArcGISProject("CURRENT")
active_map = aprx.activeMap

In [ ]:
# ── 1. ADD CSV TO MAP ──────────────────────────────────────────────────────────
csv_layer = active_map.addDataFromPath(csv_in)
print(f"Added to map: {csv_layer.name}")

In [ ]:
# ── 2. COORDINATE TABLE TO 2-POINT LINE ───────────────────────────────────────
arcpy.management.XYToLine(
    in_table          = csv_layer,
    out_featureclass  = line_wgs84,
    startx_field      = "start_longitude_DD",
    starty_field      = "start_latitude_DD",
    endx_field        = "end_longitude_DD",
    endy_field        = "end_latitude_DD",
    line_type         = "GEODESIC",
    id_field          = None,
    spatial_reference = arcpy.SpatialReference(4326)
)
print(f"Created: {line_wgs84}")

In [ ]:
# ── 3. PROJECT LINE ──────────────────────────────────────────────────────────
arcpy.management.Project(
    in_dataset              = line_wgs84,
    out_dataset             = line_nad83,
    out_coor_system         = arcpy.SpatialReference("NAD 1983 UTM Zone 20N"),
    transform_method        = geog_transform,
    in_coor_system          = arcpy.SpatialReference(4326),
    preserve_shape          = "NO_PRESERVE_SHAPE"
)
print(f"Projected: {line_nad83}")

In [ ]:
# ── 4. XY TABLE TO POINT (invisible calc layer) ───────────────────────────────
arcpy.management.XYTableToPoint(
    in_table          = csv_layer,
    out_feature_class = pts_wgs84,
    x_field           = "av_longitude_DD",
    y_field           = "av_latitude_DD",
    z_field           = None,
    coordinate_system = arcpy.SpatialReference(4326)
)
print(f"Created: {pts_wgs84}")

In [ ]:
# ── 5. PROJECT POINTS ──────────────────────────────────────────────────────────
arcpy.management.Project(
    in_dataset              = pts_wgs84,
    out_dataset             = pts_nad83,
    out_coor_system         = arcpy.SpatialReference("NAD 1983 UTM Zone 20N"),
    transform_method        = geog_transform,
    in_coor_system          = arcpy.SpatialReference(4326),
    preserve_shape          = "NO_PRESERVE_SHAPE"
)
print(f"Projected: {pts_nad83}")